# release16_repaired_bc_warm_start_official_replay_v1

Warm-start official replay for the repaired release16 checkpoint-distillation BC handoff. The prior BC repair screen selected `release128_cb995_obj000` because it fixed all required supervised release-gate slices. This notebook tests whether that repair transfers to official daily-path walk-forward behavior.

No actor checkpoint is retained or promoted from this notebook.


In [1]:

from __future__ import annotations

import json
import math
import random
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").exists() and (candidate / "src/soxl_quant").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics, max_drawdown_interval_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.metric_reconciliation import build_warm_start_replay_segment, slice_warm_start_daily_path
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, flatten_feature_frame, init_qlib, segment_frame
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = "release16_repaired_bc_warm_start_official_replay_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
BC_REPAIR_CARD = ROOT / "runtime/model_selection/card/release16_checkpoint_distillation_bc_repair_screen_v1"
COLD_OFFICIAL_CARD = ROOT / "runtime/model_selection/card/release16_release_gate_bc_official_replay_v1"
CONTRACT_CARD = ROOT / "runtime/model_selection/card/release16_checkpoint_distillation_ytd_alignment_contract_v1"
TARGET_CONTRACT = CONTRACT_CARD / "distillation_target_contract.csv"
REFERENCE_METRICS = ROOT / "runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/candidate_metrics.csv"
RETENTION_GATE = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
SEED = 20260615
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 20.0
STRICT_MAX_DD = 15.0
CARD.mkdir(parents=True, exist_ok=True)
DAILY_DIR = CARD / "official_daily_paths"
DAILY_DIR.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    "wf_2020": {
        "target_fold": "wf_2020_train",
        "train": ("2010-03-11", "2018-12-31"),
        "valid": ("2019-01-01", "2019-12-31"),
        "test": ("2020-01-01", "2020-12-31"),
    },
    "wf_2022": {
        "target_fold": "wf_2022_train",
        "train": ("2010-03-11", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test": ("2022-01-01", "2022-12-31"),
    },
    "wf_2023_2024": {
        "target_fold": "wf_2023_2024_train",
        "train": ("2010-03-11", "2022-12-31"),
        "valid": ("2023-01-01", "2023-12-31"),
        "test": ("2024-01-01", "2024-12-31"),
    },
}

CANDIDATES = [
    {
        "candidate": "current_readiness_baseline",
        "role": "direct_baseline",
        "train_config": StateObjectiveBCTrainingConfig(steps=400, lr=0.001, batch_size=128, seed=SEED),
        "weight_config": StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=6.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=16.0,
            ordinary_multiplier=1.0,
            objective_weight_power=0.20,
            class_balanced_beta=0.99,
            weight_floor=0.05,
            weight_cap=35.0,
            normalize_mean=True,
        ),
    },
    {
        "candidate": "release64_cb99_obj020",
        "role": "prior_official_reference",
        "train_config": StateObjectiveBCTrainingConfig(steps=500, lr=0.001, batch_size=128, seed=SEED + 1),
        "weight_config": StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=6.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=64.0,
            ordinary_multiplier=1.0,
            objective_weight_power=0.20,
            class_balanced_beta=0.99,
            weight_floor=0.05,
            weight_cap=80.0,
            normalize_mean=True,
        ),
    },
    {
        "candidate": "release128_cb995_obj000",
        "role": "selected_repaired_bc",
        "train_config": StateObjectiveBCTrainingConfig(steps=500, lr=0.001, batch_size=128, seed=SEED + 2),
        "weight_config": StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=6.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=128.0,
            ordinary_multiplier=1.0,
            objective_weight_power=0.0,
            class_balanced_beta=0.995,
            weight_floor=0.05,
            weight_cap=140.0,
            normalize_mean=True,
        ),
    },
]
SELECTED_CANDIDATE = "release128_cb995_obj000"
SLICE_COLS = ("is_rebound_cluster", "is_downtrend_state", "release_gate_state", "drawdown_control_state")


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False).astype(bool)
    if series.dtype == object:
        return series.fillna(False).astype(str).str.lower().isin({"true", "1", "yes", "y", "t"})
    return series.fillna(False).astype(bool)


def load_targets(path: Path) -> pd.DataFrame:
    targets = pd.read_csv(path)
    targets["datetime"] = pd.to_datetime(targets["datetime"]).dt.normalize()
    for col in ("training_label_allowed", *SLICE_COLS):
        if col in targets.columns:
            targets[col] = to_bool_series(targets[col])
    return targets.sort_values(["fold", "datetime"]).reset_index(drop=True)


def fold_subset(targets: pd.DataFrame, spec: dict[str, Any]) -> pd.DataFrame:
    dates = pd.to_datetime(targets["datetime"]).dt.normalize()
    fold_targets = targets.loc[
        targets["fold"].astype(str).eq(str(spec["target_fold"]))
        & (dates >= pd.Timestamp(spec["train"][0]).normalize())
        & (dates <= pd.Timestamp(spec["train"][1]).normalize())
    ].copy()
    if "training_label_allowed" in fold_targets.columns:
        fold_targets = fold_targets[fold_targets["training_label_allowed"].fillna(False).astype(bool)].copy()
    return fold_targets.sort_values("datetime").reset_index(drop=True)


def gate_loss(relative_return: float, max_dd: float, *, max_dd_target: float = TARGET_MAX_DD) -> float:
    return float(max(0.0, TARGET_RELATIVE_RETURN - relative_return) + 10.0 * max(0.0, max_dd - max_dd_target))


def relative_path(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

print("root", ROOT)
print("card", CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_repaired_bc_warm_start_official_replay_v1


## Governance

- Canonical baseline: `soxlbacktest_policy_reimplementation_v1` converted to qlib walk-forward reference.
- Direct comparison: `release16_checkpoint_distillation_bc_repair_screen_v1` and prior release64 official replay.
- Role: `diagnose` official replay transfer before PPO or actor retention.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Changed layer: `model_selection` / `bc_sample_weighting` / official replay gate.
- Acceptance: selected repaired BC must pass `relative >= +30pp` and `MaxDD <= 20%` on every warm-start official fold. Strict `<=15%` is recorded as risk context. No promotion is allowed here.


In [2]:

profile = load_profile(PROFILE_NAME)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
action_levels = list(action_levels_for_profile(profile))
targets = load_targets(TARGET_CONTRACT)
repair_summary = load_json(BC_REPAIR_CARD / "summary.json")
cold_official_summary = load_json(COLD_OFFICIAL_CARD / "summary.json")
retention_gate = load_json(RETENTION_GATE)
reference_metrics = pd.read_csv(REFERENCE_METRICS) if REFERENCE_METRICS.exists() else pd.DataFrame()

scope_audit = {
    "rows": int(len(targets)),
    "training_label_allowed_rows": int(targets.get("training_label_allowed", pd.Series(False, index=targets.index)).sum()),
    "blocked_training_label_rows": int((~targets.get("training_label_allowed", pd.Series(False, index=targets.index))).sum()),
    "target_scope_values": sorted(targets.get("target_scope", pd.Series(dtype=object)).dropna().astype(str).unique().tolist()),
    "label_role_values": sorted(targets.get("label_role", pd.Series(dtype=object)).dropna().astype(str).unique().tolist()),
    "oos_guard_values": sorted(targets.get("oos_or_ytd_direct_label_guard", pd.Series(dtype=object)).dropna().astype(str).unique().tolist()),
    "min_date": str(targets["datetime"].min().date()),
    "max_date": str(targets["datetime"].max().date()),
}
scope_audit["target_scope_gate"] = (
    scope_audit["blocked_training_label_rows"] == 0
    and scope_audit["target_scope_values"] == ["fold_local"]
    and scope_audit["oos_guard_values"] == ["allowed_train_dates_only"]
)
pd.DataFrame([scope_audit]).to_csv(CARD / "target_scope_audit.csv", index=False)

config_rows = []
for cfg in CANDIDATES:
    row = {"candidate": cfg["candidate"], "role": cfg["role"]}
    row.update({f"train_{k}": v for k, v in asdict(cfg["train_config"]).items()})
    row.update(asdict(cfg["weight_config"]))
    config_rows.append(row)
config_grid = pd.DataFrame(config_rows)
config_grid.to_csv(CARD / "config_grid.csv", index=False)

print("profile", profile.name, "feature_count", len(feature_columns), "action_levels", action_levels)
print("repair_decision", repair_summary.get("decision"), "selected", repair_summary.get("selected_config"))
print("cold_official_decision", cold_official_summary.get("decision"))
print("scope_audit", scope_audit)
display(config_grid)


profile sparse_release_bc_ppo_screen_v1 feature_count 59 action_levels [np.float32(0.0), np.float32(0.1), np.float32(0.2), np.float32(0.3), np.float32(0.4), np.float32(0.5), np.float32(0.6), np.float32(0.7), np.float32(0.8), np.float32(0.9), np.float32(1.0)]
repair_decision bc_repair_weighting_ready_for_checkpoint_replay_no_promotion selected release128_cb995_obj000
cold_official_decision release_gate_weighted_bc_official_replay_failed_no_promotion
scope_audit {'rows': 3731, 'training_label_allowed_rows': 3731, 'blocked_training_label_rows': 0, 'target_scope_values': ['fold_local'], 'label_role_values': ['fold_local_train_label'], 'oos_guard_values': ['allowed_train_dates_only'], 'min_date': '2010-03-11', 'max_date': '2022-12-30', 'target_scope_gate': True}


,candidate,role,train_steps,train_lr,train_batch_size,train_seed,rebound_multiplier,downtrend_multiplier,release_gate_multiplier,ordinary_multiplier,objective_weight_power,class_balanced_beta,weight_floor,weight_cap,normalize_mean
0,current_readiness_baseline,direct_baseline,400,0.001,128,20260615,6.0,2.0,16.0,1.0,0.2,0.990,0.05,35.0,True
1,release64_cb99_obj020,prior_official_reference,500,0.001,128,20260616,6.0,2.0,64.0,1.0,0.2,0.990,0.05,80.0,True
2,release128_cb995_obj000,selected_repaired_bc,500,0.001,128,20260617,6.0,2.0,128.0,1.0,0.0,0.995,0.05,140.0,True


In [3]:

init_qlib(QLIB_DATA_PATH)
fold_cache: dict[str, dict[str, Any]] = {}
warm_rows: list[dict[str, Any]] = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, "train")
    feature_full = flatten_feature_frame(dataset.handler.feature_frame).droplevel("instrument").copy()
    price_full = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
    warm_seg = build_warm_start_replay_segment(feature_full.index, eval_seg=spec["test"], lookback=LOOKBACK)
    fold_cache[fold_name] = {
        "dataset": dataset,
        "train_df": train_df,
        "feature_full": feature_full,
        "price_full": price_full,
        "warm_seg": warm_seg,
    }
    warm_rows.append(
        {
            "fold": fold_name,
            "train_start": spec["train"][0],
            "train_end": spec["train"][1],
            "test_start": spec["test"][0],
            "test_end": spec["test"][1],
            "feature_start": str(feature_full.index.min().date()),
            "feature_end": str(feature_full.index.max().date()),
            "warm_start": warm_seg[0],
            "warm_end": warm_seg[1],
            "lookback": LOOKBACK,
        }
    )
warm_start_segments = pd.DataFrame(warm_rows)
warm_start_segments.to_csv(CARD / "warm_start_segments.csv", index=False)
display(warm_start_segments)


[86463:MainThread](2026-06-15 05:50:49,481) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[86463:MainThread](2026-06-15 05:50:49,886) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[86463:MainThread](2026-06-15 05:50:49,886) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[86463:MainThread](2026-06-15 05:50:50,817) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[86463:MainThread](2026-06-15 05:50:50,818) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

[86463:MainThread](2026-06-15 05:50:51,828) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


[86463:MainThread](2026-06-15 05:50:53,028) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[86463:MainThread](2026-06-15 05:50:53,028) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[86463:MainThread](2026-06-15 05:50:53,029) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


,fold,train_start,train_end,test_start,test_end,feature_start,feature_end,warm_start,warm_end,lookback
0,wf_2020,2010-03-11,2018-12-31,2020-01-01,2020-12-31,2011-03-11,2020-12-31,2019-10-07,2020-12-31,60
1,wf_2022,2010-03-11,2020-12-31,2022-01-01,2022-12-31,2011-03-11,2022-12-30,2021-10-07,2022-12-30,60
2,wf_2023_2024,2010-03-11,2022-12-31,2024-01-01,2024-12-31,2011-03-11,2024-12-31,2023-10-05,2024-12-31,60


In [4]:

metric_rows: list[dict[str, Any]] = []
fit_rows: list[dict[str, Any]] = []
slice_frames: list[pd.DataFrame] = []
action_frames: list[pd.DataFrame] = []
drawdown_rows: list[dict[str, Any]] = []
daily_manifest: list[dict[str, str]] = []

for candidate_idx, cfg in enumerate(CANDIDATES):
    candidate = str(cfg["candidate"])
    print("candidate", candidate)
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        cache = fold_cache[fold_name]
        fold_targets = fold_subset(targets, spec)
        print("  fold", fold_name, "targets", len(fold_targets))
        if fold_targets.empty:
            metric_rows.append({"candidate": candidate, "fold": fold_name, "status": "skipped_no_targets"})
            continue

        set_seed(SEED + candidate_idx * 101 + len(fold_name))
        fit = fit_state_objective_bc_actor_from_frame(
            cache["train_df"],
            targets=fold_targets,
            profile=profile,
            train_config=cfg["train_config"],
            sample_weight_config=cfg["weight_config"],
            sample_size=0,
            rng_seed=SEED,
        )
        fit_rows.append(
            {
                "candidate": candidate,
                "role": cfg["role"],
                "fold": fold_name,
                "target_rows": int(len(fold_targets)),
                "bc_samples": int(len(fit.actions)),
                "before_accuracy": float(fit.metrics.get("before_accuracy", np.nan)),
                "after_accuracy": float(fit.metrics.get("after_accuracy", np.nan)),
                "after_loss": float(fit.metrics.get("after_loss", np.nan)),
                "train_last_loss": float(fit.metrics.get("train_last_loss", np.nan)),
                "weight_mean": float(fit.metrics.get("weight_mean", np.nan)),
                "weight_max": float(fit.metrics.get("weight_max", np.nan)),
            }
        )
        bc_dataset = build_state_objective_bc_dataset_from_frame(
            cache["train_df"],
            targets=fold_targets,
            profile=profile,
            sample_size=0,
            rng_seed=SEED,
        )
        slice_metrics = evaluate_bc_actor_slice_fit(
            fit.actor,
            fit.obs,
            fit.actions,
            bc_dataset.metadata,
            slice_cols=SLICE_COLS,
        )
        slice_metrics.insert(0, "fold", fold_name)
        slice_metrics.insert(0, "candidate", candidate)
        slice_frames.append(slice_metrics)

        action_dist = fit.action_distribution.copy()
        action_dist.insert(0, "fold", fold_name)
        action_dist.insert(0, "candidate", candidate)
        action_frames.append(action_dist)

        bt = backtest_profile(
            fit.actor,
            cache["feature_full"],
            cache["price_full"],
            position_cap_config=profile.position_cap,
            position_budget_config=profile.position_budget,
            execution_config=profile.execution,
            asset_idx=0,
            feature_columns=feature_columns,
            model_config=profile.model,
            mode_router_config=profile.mode_router,
            test_seg=cache["warm_seg"],
        )
        warm_daily = slice_warm_start_daily_path(
            portfolio_df=bt["portfolio"],
            price_series=cache["price_full"],
            eval_seg=spec["test"],
            position_history=bt.get("position_history", []),
            trade_signals=bt.get("trade_signals", []),
        )
        daily_path = DAILY_DIR / f"{candidate}_{fold_name}_warm_start_official_daily_path.csv"
        warm_daily.to_csv(daily_path, index=False)
        daily_manifest.append({"candidate": candidate, "fold": fold_name, "path": relative_path(daily_path)})
        metrics = daily_path_metrics(warm_daily)
        interval = max_drawdown_interval_metrics(warm_daily)
        drawdown_rows.append({"candidate": candidate, "fold": fold_name, **interval})

        row = {
            "candidate": candidate,
            "role": cfg["role"],
            "fold": fold_name,
            "status": "evaluated",
            "target_rows": int(len(fold_targets)),
            "bc_samples": int(len(fit.actions)),
            "bc_after_accuracy": float(fit.metrics.get("after_accuracy", np.nan)),
            "bc_after_loss": float(fit.metrics.get("after_loss", np.nan)),
            "requested_start": spec["test"][0],
            "requested_end": spec["test"][1],
            "warm_start": cache["warm_seg"][0],
            "warm_end": cache["warm_seg"][1],
            "official_warm_return": float(metrics["return"]),
            "official_warm_buyhold_return": float(metrics["buy_and_hold_return"]),
            "official_warm_relative_return": float(metrics["relative_return"]),
            "official_warm_max_dd": float(metrics["max_dd"]),
            "official_warm_avg_pos": float(metrics["avg_pos"]),
            "daily_path_rows": int(metrics["rows"]),
            "trades": int(bt.get("trades", 0)),
            "full_gate_20dd": bool(float(metrics["relative_return"]) >= TARGET_RELATIVE_RETURN and float(metrics["max_dd"]) <= TARGET_MAX_DD),
            "strict_gate_15dd": bool(float(metrics["relative_return"]) >= TARGET_RELATIVE_RETURN and float(metrics["max_dd"]) <= STRICT_MAX_DD),
            "gate_loss_20dd": gate_loss(float(metrics["relative_return"]), float(metrics["max_dd"]), max_dd_target=TARGET_MAX_DD),
            "gate_loss_15dd": gate_loss(float(metrics["relative_return"]), float(metrics["max_dd"]), max_dd_target=STRICT_MAX_DD),
            "daily_path": relative_path(daily_path),
        }
        ref = reference_metrics[
            reference_metrics.get("candidate", pd.Series(dtype=str)).astype(str).eq("legacy_default")
            & reference_metrics.get("window", pd.Series(dtype=str)).astype(str).eq(fold_name)
        ] if not reference_metrics.empty else pd.DataFrame()
        if not ref.empty:
            row["soxlbacktest_legacy_return_pct"] = float(ref["rulebase_return_pct"].iloc[0])
            row["soxlbacktest_legacy_max_dd_pct"] = float(ref["rulebase_max_drawdown_pct"].iloc[0])
            row["soxlbacktest_legacy_avg_pos_pct"] = float(ref["average_position_pct"].iloc[0])
        metric_rows.append(row)

bc_fit_metrics = pd.DataFrame(fit_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
bc_slice_metrics = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
bc_action_distribution = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()
warm_start_official_metrics = pd.DataFrame(metric_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
drawdown_intervals = pd.DataFrame(drawdown_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
daily_manifest_df = pd.DataFrame(daily_manifest).sort_values(["candidate", "fold"]).reset_index(drop=True)

bc_fit_metrics.to_csv(CARD / "bc_fit_metrics.csv", index=False)
bc_slice_metrics.to_csv(CARD / "bc_slice_metrics.csv", index=False)
bc_action_distribution.to_csv(CARD / "bc_action_distribution.csv", index=False)
warm_start_official_metrics.to_csv(CARD / "warm_start_official_metrics.csv", index=False)
drawdown_intervals.to_csv(CARD / "drawdown_intervals.csv", index=False)
daily_manifest_df.to_csv(CARD / "daily_manifest.csv", index=False)

display(bc_fit_metrics)
display(bc_slice_metrics[bc_slice_metrics["slice"].eq("release_gate_state")])
display(warm_start_official_metrics)
display(drawdown_intervals)


candidate current_readiness_baseline
  fold wf_2020 targets 2218


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[86463:MainThread](2026-06-15 05:50:58,167) WARNING - qlib.data - [data.py:665] - load calendar error: freq=day, future=True; return current calendar!


[86463:MainThread](2026-06-15 05:50:58,168) WARNING - qlib.data - [data.py:668] - You can get future calendar by referring to the following document: https://github.com/microsoft/qlib/blob/main/scripts/data_collector/contrib/README.md


[86463:MainThread](2026-06-15 05:50:58,170) WARNING - qlib.BaseExecutor - [executor.py:121] - `common_infra` is not set for <qlib.backtest.executor.SimulatorExecutor object at 0x122ba6410>


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

,candidate,role,fold,target_rows,bc_samples,before_accuracy,after_accuracy,after_loss,train_last_loss,weight_mean,weight_max
0,current_readiness_baseline,direct_baseline,wf_2020,2218,1904,0.305147,0.723214,1.184816,1.152152,1.0,14.726513
1,current_readiness_baseline,direct_baseline,wf_2022,253,252,0.138889,0.841270,1.061176,0.981473,1.0,6.894364
2,current_readiness_baseline,direct_baseline,wf_2023_2024,756,755,0.074172,0.772185,1.134638,1.128318,1.0,9.615528
3,release128_cb995_obj000,selected_repaired_bc,wf_2020,2218,1904,0.117647,0.597164,1.309246,1.033800,1.0,43.585609
4,release128_cb995_obj000,selected_repaired_bc,wf_2022,253,252,0.019841,0.718254,1.183019,0.976954,1.0,27.901085
5,release128_cb995_obj000,selected_repaired_bc,wf_2023_2024,756,755,0.023841,0.752318,1.158815,1.186110,1.0,31.258448
6,release64_cb99_obj020,prior_official_reference,wf_2020,2218,1904,0.181197,0.713235,1.204745,1.045434,1.0,24.481781
7,release64_cb99_obj020,prior_official_reference,wf_2022,253,252,0.019841,0.829365,1.077159,0.967088,1.0,12.192379
8,release64_cb99_obj020,prior_official_reference,wf_2023_2024,756,755,0.041060,0.753642,1.150103,1.150085,1.0,17.890619


,candidate,fold,slice,samples,accuracy,loss,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share
4,current_readiness_baseline,wf_2020,release_gate_state,32,0.750000,1.152201,3,3,4,0.843750,4,0.718750
9,current_readiness_baseline,wf_2022,release_gate_state,8,1.000000,0.905731,4,4,4,0.375000,4,0.375000
14,current_readiness_baseline,wf_2023_2024,release_gate_state,11,0.363636,1.540462,4,2,3,0.454545,4,0.727273
19,release64_cb99_obj020,wf_2020,release_gate_state,32,0.906250,1.003206,3,3,4,0.843750,4,0.750000
24,release64_cb99_obj020,wf_2022,release_gate_state,8,1.000000,0.905055,4,4,4,0.375000,4,0.375000
29,release64_cb99_obj020,wf_2023_2024,release_gate_state,11,0.727273,1.177735,4,2,3,0.454545,4,0.545455
34,release128_cb995_obj000,wf_2020,release_gate_state,32,1.000000,0.907483,3,3,4,0.843750,4,0.843750
39,release128_cb995_obj000,wf_2022,release_gate_state,8,1.000000,0.904967,4,4,4,0.375000,4,0.375000
44,release128_cb995_obj000,wf_2023_2024,release_gate_state,11,0.818182,1.093103,4,2,3,0.454545,3,0.545455


,candidate,role,fold,status,target_rows,bc_samples,bc_after_accuracy,bc_after_loss,requested_start,requested_end,...,daily_path_rows,trades,full_gate_20dd,strict_gate_15dd,gate_loss_20dd,gate_loss_15dd,daily_path,soxlbacktest_legacy_return_pct,soxlbacktest_legacy_max_dd_pct,soxlbacktest_legacy_avg_pos_pct
0,current_readiness_baseline,direct_baseline,wf_2020,evaluated,2218,1904,0.723214,1.184816,2020-01-01,2020-12-31,...,252,50,False,False,218.940832,268.940832,runtime/model_selection/card/release16_repaire...,220.777205,54.275144,79.102332
1,current_readiness_baseline,direct_baseline,wf_2022,evaluated,253,252,0.841270,1.061176,2022-01-01,2022-12-31,...,250,45,False,False,42.244376,92.244376,runtime/model_selection/card/release16_repaire...,-79.396371,84.918053,86.692988
2,current_readiness_baseline,direct_baseline,wf_2023_2024,evaluated,756,755,0.772185,1.134638,2024-01-01,2024-12-31,...,251,49,False,False,5.138403,5.138403,runtime/model_selection/card/release16_repaire...,2.935118,32.461810,31.043492
3,release128_cb995_obj000,selected_repaired_bc,wf_2020,evaluated,2218,1904,0.597164,1.309246,2020-01-01,2020-12-31,...,252,51,False,False,198.417153,248.417153,runtime/model_selection/card/release16_repaire...,220.777205,54.275144,79.102332
4,release128_cb995_obj000,selected_repaired_bc,wf_2022,evaluated,253,252,0.718254,1.183019,2022-01-01,2022-12-31,...,250,44,False,False,60.216277,110.216277,runtime/model_selection/card/release16_repaire...,-79.396371,84.918053,86.692988
5,release128_cb995_obj000,selected_repaired_bc,wf_2023_2024,evaluated,756,755,0.752318,1.158815,2024-01-01,2024-12-31,...,251,47,False,False,3.518821,20.822880,runtime/model_selection/card/release16_repaire...,2.935118,32.461810,31.043492
6,release64_cb99_obj020,prior_official_reference,wf_2020,evaluated,2218,1904,0.713235,1.204745,2020-01-01,2020-12-31,...,252,50,False,False,99.096289,149.096289,runtime/model_selection/card/release16_repaire...,220.777205,54.275144,79.102332
7,release64_cb99_obj020,prior_official_reference,wf_2022,evaluated,253,252,0.829365,1.077159,2022-01-01,2022-12-31,...,250,43,False,False,46.253203,96.253203,runtime/model_selection/card/release16_repaire...,-79.396371,84.918053,86.692988
8,release64_cb99_obj020,prior_official_reference,wf_2023_2024,evaluated,756,755,0.753642,1.150103,2024-01-01,2024-12-31,...,251,49,True,False,0.000000,39.648344,runtime/model_selection/card/release16_repaire...,2.935118,32.461810,31.043492


,candidate,fold,peak_date,trough_date,max_dd,interval_return,interval_days,interval_avg_pos,interval_min_pos,interval_max_pos,interval_trades
0,current_readiness_baseline,wf_2020,2020-02-19,2020-03-20,34.734407,-34.734407,23,0.257027,0.127230,0.452000,2
1,current_readiness_baseline,wf_2022,2022-01-03,2022-11-03,24.224438,-24.224438,212,0.077734,0.000000,0.374746,31
2,current_readiness_baseline,wf_2023_2024,2024-07-10,2024-12-19,13.917934,-13.917934,115,0.118758,0.000000,0.487350,17
3,release128_cb995_obj000,wf_2020,2020-02-19,2020-03-20,34.340318,-34.340318,23,0.268029,0.126551,0.450000,3
4,release128_cb995_obj000,wf_2022,2022-01-03,2022-12-06,26.021628,-26.021628,234,0.111531,0.000000,0.710440,33
5,release128_cb995_obj000,wf_2023_2024,2024-03-07,2024-04-19,16.730406,-16.730406,31,0.326707,0.091506,0.486341,5
6,release64_cb99_obj020,wf_2020,2020-02-19,2020-03-20,26.706749,-26.706749,23,0.228073,0.122797,0.289424,3
7,release64_cb99_obj020,wf_2022,2022-01-03,2022-11-03,24.625320,-24.625320,212,0.089161,0.000000,0.370085,27
8,release64_cb99_obj020,wf_2023_2024,2024-03-07,2024-04-19,18.964834,-18.964834,31,0.413778,0.220573,0.483086,4


In [5]:

summary_rows: list[dict[str, Any]] = []
for candidate, group in warm_start_official_metrics.groupby("candidate"):
    evaluated = group[group["status"].eq("evaluated")].copy()
    role = str(evaluated["role"].iloc[0]) if not evaluated.empty else ""
    row = {
        "candidate": candidate,
        "role": role,
        "evaluated_fold_count": int(len(evaluated)),
        "all_folds_evaluated": bool(set(evaluated["fold"].astype(str)) == set(WALK_FORWARD_WINDOWS)),
        "full_gate_20dd_count": int(evaluated["full_gate_20dd"].fillna(False).astype(bool).sum()),
        "strict_gate_15dd_count": int(evaluated["strict_gate_15dd"].fillna(False).astype(bool).sum()),
        "all_full_gate_20dd": bool(len(evaluated) == len(WALK_FORWARD_WINDOWS) and evaluated["full_gate_20dd"].fillna(False).astype(bool).all()),
        "all_strict_gate_15dd": bool(len(evaluated) == len(WALK_FORWARD_WINDOWS) and evaluated["strict_gate_15dd"].fillna(False).astype(bool).all()),
        "min_relative_return": float(evaluated["official_warm_relative_return"].min()) if not evaluated.empty else math.nan,
        "max_max_dd": float(evaluated["official_warm_max_dd"].max()) if not evaluated.empty else math.nan,
        "mean_relative_return": float(evaluated["official_warm_relative_return"].mean()) if not evaluated.empty else math.nan,
        "mean_avg_pos": float(evaluated["official_warm_avg_pos"].mean()) if not evaluated.empty else math.nan,
        "total_gate_loss_20dd": float(evaluated["gate_loss_20dd"].sum()) if not evaluated.empty else math.nan,
        "total_gate_loss_15dd": float(evaluated["gate_loss_15dd"].sum()) if not evaluated.empty else math.nan,
        "failed_folds_20dd": sorted(evaluated.loc[~evaluated["full_gate_20dd"].fillna(False).astype(bool), "fold"].astype(str).tolist()),
    }
    for fold_name in WALK_FORWARD_WINDOWS:
        rec = evaluated[evaluated["fold"].astype(str).eq(fold_name)]
        if not rec.empty:
            row[f"{fold_name}_relative_return"] = float(rec["official_warm_relative_return"].iloc[0])
            row[f"{fold_name}_max_dd"] = float(rec["official_warm_max_dd"].iloc[0])
            row[f"{fold_name}_avg_pos"] = float(rec["official_warm_avg_pos"].iloc[0])
    summary_rows.append(row)

candidate_summary = pd.DataFrame(summary_rows).sort_values(
    ["all_full_gate_20dd", "full_gate_20dd_count", "min_relative_return", "max_max_dd", "total_gate_loss_20dd"],
    ascending=[False, False, False, True, True],
).reset_index(drop=True)
candidate_summary.to_csv(CARD / "candidate_summary.csv", index=False)

selected_metrics = warm_start_official_metrics[warm_start_official_metrics["candidate"].eq(SELECTED_CANDIDATE)].copy()
baseline_metrics = warm_start_official_metrics[warm_start_official_metrics["candidate"].eq("current_readiness_baseline")].copy()
delta_rows = []
for fold_name in WALK_FORWARD_WINDOWS:
    sel = selected_metrics[selected_metrics["fold"].eq(fold_name)]
    base = baseline_metrics[baseline_metrics["fold"].eq(fold_name)]
    if sel.empty or base.empty:
        continue
    s = sel.iloc[0]
    b = base.iloc[0]
    delta_rows.append(
        {
            "fold": fold_name,
            "selected_candidate": SELECTED_CANDIDATE,
            "selected_relative_return": float(s["official_warm_relative_return"]),
            "baseline_relative_return": float(b["official_warm_relative_return"]),
            "relative_delta_pp": float(s["official_warm_relative_return"] - b["official_warm_relative_return"]),
            "selected_max_dd": float(s["official_warm_max_dd"]),
            "baseline_max_dd": float(b["official_warm_max_dd"]),
            "max_dd_delta_pp": float(s["official_warm_max_dd"] - b["official_warm_max_dd"]),
            "selected_avg_pos": float(s["official_warm_avg_pos"]),
            "baseline_avg_pos": float(b["official_warm_avg_pos"]),
            "avg_pos_delta": float(s["official_warm_avg_pos"] - b["official_warm_avg_pos"]),
            "selected_full_gate_20dd": bool(s["full_gate_20dd"]),
            "baseline_full_gate_20dd": bool(b["full_gate_20dd"]),
        }
    )
selected_vs_baseline_delta = pd.DataFrame(delta_rows)
selected_vs_baseline_delta.to_csv(CARD / "selected_vs_baseline_delta.csv", index=False)

selected_summary = candidate_summary[candidate_summary["candidate"].eq(SELECTED_CANDIDATE)]
selected_row = selected_summary.iloc[0].to_dict() if not selected_summary.empty else {}
selected_all_20dd = bool(selected_row.get("all_full_gate_20dd", False))
selected_all_15dd = bool(selected_row.get("all_strict_gate_15dd", False))

decision = (
    "repaired_bc_warm_start_official_replay_passed_20dd_no_promotion"
    if selected_all_20dd
    else "repaired_bc_warm_start_official_replay_failed_no_promotion"
)
next_mechanism = (
    "run_drift_aware_checkpoint_or_short_ppo_screen_with_repaired_bc_initialization"
    if selected_all_20dd
    else "repair_replay_objective_with_drawdown_drift_and_trend_participation_before_ppo"
)

selection = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "next_mechanism": next_mechanism,
    "selected_candidate": SELECTED_CANDIDATE,
    "selected_summary": selected_row,
    "selected_all_20dd": selected_all_20dd,
    "selected_all_15dd": selected_all_15dd,
    "candidate_summary": candidate_summary.to_dict("records"),
    "selected_vs_baseline_delta": selected_vs_baseline_delta.to_dict("records"),
    "promotion_allowed": False,
    "actor_retention_allowed": False,
}
write_json(CARD / "selection.json", selection)

display(candidate_summary)
display(selected_vs_baseline_delta)
print("decision", decision, "next", next_mechanism)


,candidate,role,evaluated_fold_count,all_folds_evaluated,full_gate_20dd_count,strict_gate_15dd_count,all_full_gate_20dd,all_strict_gate_15dd,min_relative_return,max_max_dd,...,failed_folds_20dd,wf_2020_relative_return,wf_2020_max_dd,wf_2020_avg_pos,wf_2022_relative_return,wf_2022_max_dd,wf_2022_avg_pos,wf_2023_2024_relative_return,wf_2023_2024_max_dd,wf_2023_2024_avg_pos
0,release64_cb99_obj020,prior_official_reference,3,True,1,0,False,False,-2.028796,26.706749,...,"[wf_2020, wf_2022]",-2.028796,26.706749,0.344693,64.650534,24.625320,0.116979,34.581169,18.964834,0.284346
1,release128_cb995_obj000,selected_repaired_bc,3,True,0,0,False,False,-25.013978,34.340318,...,"[wf_2020, wf_2022, wf_2023_2024]",-25.013978,34.340318,0.384665,62.347356,26.021628,0.110629,26.481179,16.730406,0.243529
2,current_readiness_baseline,direct_baseline,3,True,0,0,False,False,-41.596766,34.734407,...,"[wf_2020, wf_2022, wf_2023_2024]",-41.596766,34.734407,0.402347,65.866980,24.224438,0.092157,24.861597,13.917934,0.243178


,fold,selected_candidate,selected_relative_return,baseline_relative_return,relative_delta_pp,selected_max_dd,baseline_max_dd,max_dd_delta_pp,selected_avg_pos,baseline_avg_pos,avg_pos_delta,selected_full_gate_20dd,baseline_full_gate_20dd
0,wf_2020,release128_cb995_obj000,-25.013978,-41.596766,16.582788,34.340318,34.734407,-0.394089,0.384665,0.402347,-0.017683,False,False
1,wf_2022,release128_cb995_obj000,62.347356,65.866980,-3.519624,26.021628,24.224438,1.797190,0.110629,0.092157,0.018472,False,False
2,wf_2023_2024,release128_cb995_obj000,26.481179,24.861597,1.619582,16.730406,13.917934,2.812472,0.243529,0.243178,0.000352,False,False


decision repaired_bc_warm_start_official_replay_failed_no_promotion next repair_replay_objective_with_drawdown_drift_and_trend_participation_before_ppo


In [6]:

retention_summary = retention_gate.get("summary", {}) if isinstance(retention_gate, dict) else {}
output_paths = {
    "config_grid": CARD / "config_grid.csv",
    "target_scope_audit": CARD / "target_scope_audit.csv",
    "warm_start_segments": CARD / "warm_start_segments.csv",
    "bc_fit_metrics": CARD / "bc_fit_metrics.csv",
    "bc_slice_metrics": CARD / "bc_slice_metrics.csv",
    "bc_action_distribution": CARD / "bc_action_distribution.csv",
    "warm_start_official_metrics": CARD / "warm_start_official_metrics.csv",
    "candidate_summary": CARD / "candidate_summary.csv",
    "selected_vs_baseline_delta": CARD / "selected_vs_baseline_delta.csv",
    "drawdown_intervals": CARD / "drawdown_intervals.csv",
    "daily_manifest": CARD / "daily_manifest.csv",
    "official_daily_paths": DAILY_DIR,
    "selection": CARD / "selection.json",
    "summary": CARD / "summary.json",
    "conclusion": CARD / "conclusion.md",
}

summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "next_mechanism": next_mechanism,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "role": "diagnose_repaired_bc_official_replay",
    "canonical_baseline": "soxlbacktest_policy_reimplementation_v1",
    "direct_comparison_baseline": [
        "release16_checkpoint_distillation_bc_repair_screen_v1",
        "release16_release_gate_bc_official_replay_v1",
    ],
    "target_window": list(WALK_FORWARD_WINDOWS),
    "changed_layer": ["model_selection", "bc_sample_weighting", "warm_start_official_replay"],
    "external_reference": "converted SOXLBacktest qlib walk-forward buy/sell baseline",
    "hypothesis": "The repaired release-gate BC weighting should transfer from supervised slice fit to official warm-start walk-forward behavior.",
    "acceptance_criteria": [
        "selected repaired BC replayed on wf_2020, wf_2022, and wf_2023_2024",
        "selected repaired BC passes relative_return >= 30pp and MaxDD <= 20% on every fold",
        "strict <=15% MaxDD gate recorded as risk context",
        "no actor retained or promoted from this diagnostic replay",
    ],
    "expected_tradeoff": "Higher release-state fit may increase trend participation but can worsen stress-fold drawdown.",
    "failure_mode": "Supervised release-gate slice repair may not transfer to official daily path gates, especially wf_2022 drawdown and wf_2023_2024 relative return.",
    "comparison_table_row_key": EXPERIMENT,
    "closest_historical_success_case": "release16_checkpoint_distillation_bc_repair_screen_v1 fixed required supervised release-gate slices for 3/3 folds.",
    "closest_historical_failure_case": "release16_release_gate_bc_official_replay_v1 fixed slice fit but failed all selected official walk-forward path gates.",
    "why_not_duplicate": "This notebook uses the newly selected release128_cb995_obj000 weighting and warm-start official replay; the previous release16 official replay used release64 and cold/eval-window replay metrics.",
    "profile": profile.name,
    "source_repair_decision": repair_summary.get("decision"),
    "source_repair_selected_config": repair_summary.get("selected_config"),
    "source_cold_official_decision": cold_official_summary.get("decision"),
    "target_scope_audit": scope_audit,
    "selected_candidate": SELECTED_CANDIDATE,
    "selected_summary": selected_row,
    "candidate_summary": candidate_summary.to_dict("records"),
    "gate_summary": {
        "all_folds_replayed": bool(candidate_summary["all_folds_evaluated"].all()) if not candidate_summary.empty else False,
        "selected_all_20dd": selected_all_20dd,
        "selected_all_15dd": selected_all_15dd,
        "selected_gate_count_20dd": int(selected_row.get("full_gate_20dd_count", 0)) if selected_row else 0,
        "selected_gate_count_15dd": int(selected_row.get("strict_gate_15dd_count", 0)) if selected_row else 0,
        "required_fold_count": len(WALK_FORWARD_WINDOWS),
    },
    "retention_gate_current": {
        "passing_profile_count": retention_summary.get("passing_profile_count"),
        "max_walkforward_drawdown_pct": retention_summary.get("max_walkforward_drawdown_pct"),
        "current_best_return_pct": retention_summary.get("current_best_return_pct"),
        "required_best_return_pct": retention_summary.get("required_best_return_pct"),
        "reference_profile": retention_summary.get("reference_profile"),
        "reference_return_pct": retention_summary.get("reference_return_pct"),
    },
    "output_paths": {key: relative_path(path) for key, path in output_paths.items()},
    "conclusion": (
        "Selected repaired BC passes the 20% warm-start official replay gate, but still has no promotion until retained actor/YTD/fixed-split review."
        if selected_all_20dd
        else "Selected repaired BC does not pass warm-start official walk-forward path gates; do not run PPO or retain an actor from this card."
    ),
}
write_json(CARD / "summary.json", summary)

lines = [
    f"# {EXPERIMENT}",
    "",
    f"Decision: `{decision}`.",
    f"Selected candidate: `{SELECTED_CANDIDATE}`.",
    f"Next mechanism: `{next_mechanism}`.",
    "",
    "No actor checkpoint is retained or promoted from this screen.",
    "",
    "Key evidence:",
]
if selected_row:
    lines.extend(
        [
            f"- Selected 20% gate count: {int(selected_row.get('full_gate_20dd_count', 0))} / {len(WALK_FORWARD_WINDOWS)}.",
            f"- Selected strict 15% gate count: {int(selected_row.get('strict_gate_15dd_count', 0))} / {len(WALK_FORWARD_WINDOWS)}.",
            f"- Selected min relative return: {float(selected_row.get('min_relative_return', math.nan)):.4f} pp.",
            f"- Selected worst MaxDD: {float(selected_row.get('max_max_dd', math.nan)):.4f}%.",
            f"- Failed folds under 20% gate: {selected_row.get('failed_folds_20dd', [])}.",
        ]
    )
lines.extend([
    "",
    "Artifacts:",
    "- `warm_start_official_metrics.csv`",
    "- `candidate_summary.csv`",
    "- `selected_vs_baseline_delta.csv`",
    "- `official_daily_paths/*.csv`",
])
(CARD / "conclusion.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
display(Markdown("\n".join(lines)))


# release16_repaired_bc_warm_start_official_replay_v1

Decision: `repaired_bc_warm_start_official_replay_failed_no_promotion`.
Selected candidate: `release128_cb995_obj000`.
Next mechanism: `repair_replay_objective_with_drawdown_drift_and_trend_participation_before_ppo`.

No actor checkpoint is retained or promoted from this screen.

Key evidence:
- Selected 20% gate count: 0 / 3.
- Selected strict 15% gate count: 0 / 3.
- Selected min relative return: -25.0140 pp.
- Selected worst MaxDD: 34.3403%.
- Failed folds under 20% gate: ['wf_2020', 'wf_2022', 'wf_2023_2024'].

Artifacts:
- `warm_start_official_metrics.csv`
- `candidate_summary.csv`
- `selected_vs_baseline_delta.csv`
- `official_daily_paths/*.csv`